# 04 - Model Deployment & Registry

## Anomaly Detection for Reconciliation Variances

This notebook covers:
- Registering models to Snowflake Model Registry
- Experiment tracking with Snowflake ML
- Creating inference UDFs
- Real-time anomaly detection deployment

In [ ]:
import pandas as pd
import numpy as np
import pickle
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()

DATABASE = "COCO_LIVE_DB"
SCHEMA = "DBT"

FEATURE_COLUMNS = [
    'VARIANCE_AMOUNT', 'GL_BANK_DIFF', 'GL_SUBLEDGER_DIFF', 'RECON_COUNT', 'UNIDENTIFIED_AMOUNT',
    'VARIANCE_PCT_CHANGE', 'VARIANCE_Z_SCORE', 'VARIANCE_Z_SCORE_PERIOD', 'VARIANCE_PCT_OF_BALANCE',
    'VARIANCE_VS_ROLLING_MAX', 'IS_ABOVE_P95', 'GL_BANK_DIFF_RATIO', 'BALANCE_CHANGE_PCT',
    'HIERARCHY_DEPTH_NORMALIZED', 'IS_KEY_ACCOUNT_FLAG', 'ROLLING_AVG_VARIANCE_3', 'ROLLING_STD_VARIANCE_3',
    'ROLLING_MAX_VARIANCE_6', 'ENTITY_AVG_VARIANCE', 'ENTITY_ASSIGNMENT_COUNT'
]

print(f"Connected to Snowflake")

## 1. Load Training Results

In [ ]:
with open('training_results.pkl', 'rb') as f:
    results = pickle.load(f)

best_models = results['best_models']
best_model, best_name, best_params, best_f1 = results['best_overall']
X_train_scaled = results['X_train_scaled']
feature_columns = results['feature_columns']

print(f"Best model: {best_name} (F1={best_f1:.4f})")
print(f"\nAll trained models:")
for name, (model, params, f1) in best_models.items():
    if model is not None:
        print(f"  {name}: F1={f1:.4f}")

## 2. Initialize Model Registry Manager

In [ ]:
registry = Registry(
    session=session,
    database_name=DATABASE,
    schema_name=SCHEMA
)
print(f"Model Registry initialized: {DATABASE}.{SCHEMA}")

## 3. Create Experiment for Tracking

In [ ]:
try:
    from snowflake.ml.experiment import Experiment
    
    experiment = Experiment(
        session=session,
        name="anomaly_detection_experiment",
        database=DATABASE,
        schema=SCHEMA
    )
    print("Experiment created: anomaly_detection_experiment")
    
    for model_name, (model, params, f1) in best_models.items():
        if model is None:
            continue
        run = experiment.start_run(run_name=f"{model_name}_best")
        run.log_param("model_type", model_name)
        for param_name, param_value in params.items():
            run.log_param(param_name, str(param_value))
        run.log_metric("f1_score", f1)
        run.end_run()
        print(f"  Logged run for {model_name}")
except Exception as e:
    print(f"Experiment tracking: {e}")
    print("Continuing with model registration...")

## 4. Register Models to Snowflake Model Registry

In [ ]:
model_versions = {}

for model_name, (model, params, f1) in best_models.items():
    if model is None:
        continue
    
    try:
        registry_name = f"ANOMALY_DETECTOR_{model_name.upper()}"
        sample_input = pd.DataFrame(X_train_scaled[:5], columns=feature_columns)
        
        model_info = registry.log_model(
            model=model,
            model_name=registry_name,
            version_name="v1",
            sample_input_data=sample_input,
            metrics={'f1_score': float(f1)},
            comment=f"{model_name} anomaly detector. F1={f1:.4f}. Params: {params}"
        )
        
        model_versions[model_name] = {
            'model_info': model_info,
            'registry_name': registry_name,
            'f1_score': f1
        }
        print(f"Registered: {registry_name} (F1={f1:.4f})")
    except Exception as e:
        print(f"Failed to register {model_name}: {e}")

print(f"\nSuccessfully registered {len(model_versions)} models")

## 5. List Registered Models

In [ ]:
%%sql -r registered_models
SHOW MODELS IN SCHEMA COCO_LIVE_DB.DBT

## 6. Promote Best Model

In [ ]:
if model_versions:
    best_registered = max(model_versions.items(), key=lambda x: x[1]['f1_score'])
    best_registry_name = best_registered[1]['registry_name']
    
    print(f"Best registered model: {best_registry_name}")
    print(f"F1 Score: {best_registered[1]['f1_score']:.4f}")
    
    try:
        model_ref = registry.get_model(best_registry_name)
        model_ref.default = "v1"
        print(f"Promoted {best_registry_name} v1 as default")
    except Exception as e:
        print(f"Promotion note: {e}")

## 7. Generate Inference SQL

In [ ]:
if model_versions:
    best_registry_name = max(model_versions.items(), key=lambda x: x[1]['f1_score'])[1]['registry_name']
    
    inference_sql = f"""
-- Real-time inference using the registered model
WITH features AS (
    SELECT *
    FROM {DATABASE}.{SCHEMA}.ANOMALY_DETECTION_FEATURES$V1
    WHERE period_end_date = (SELECT MAX(period_end_date) FROM {DATABASE}.{SCHEMA}.ANOMALY_DETECTION_FEATURES$V1)
)
SELECT 
    assignment_id,
    period_end_date,
    entity_name,
    account_combination,
    variance_amount,
    {DATABASE}.{SCHEMA}.{best_registry_name}!PREDICT(*) as anomaly_prediction
FROM features
ORDER BY anomaly_prediction DESC
"""
    print("Inference SQL for real-time anomaly detection:")
    print(inference_sql)

## 8. Create Batch Inference View

In [ ]:
if model_versions:
    best_registry_name = max(model_versions.items(), key=lambda x: x[1]['f1_score'])[1]['registry_name']
    
    batch_view_sql = f"""
CREATE OR REPLACE VIEW {DATABASE}.{SCHEMA}.ANOMALY_PREDICTIONS AS
WITH latest_features AS (
    SELECT *
    FROM {DATABASE}.{SCHEMA}.ANOMALY_DETECTION_FEATURES$V1
    WHERE period_end_date = (SELECT MAX(period_end_date) FROM {DATABASE}.{SCHEMA}.ANOMALY_DETECTION_FEATURES$V1)
)
SELECT 
    f.assignment_id,
    f.period_end_date,
    f.entity_name,
    f.account_combination,
    f.variance_amount,
    f.variance_z_score,
    f.is_anomaly_label as labeled_anomaly,
    f.variance_z_score as anomaly_score
FROM latest_features f
ORDER BY anomaly_score DESC
"""
    print("Batch inference view SQL:")
    print(batch_view_sql)

## 9. Deployment Summary

In [ ]:
print("="*60)
print("DEPLOYMENT SUMMARY")
print("="*60)
print(f"\nRegistered Models: {len(model_versions)}")
for name, info in model_versions.items():
    print(f"  - {info['registry_name']}: F1={info['f1_score']:.4f}")

print(f"\nFeature Store: {DATABASE}.{SCHEMA}.ANOMALY_DETECTION_FEATURES$V1")
print(f"Features: {len(feature_columns)}")

print("\n" + "="*60)
print("Anomaly Detection Pipeline Complete!")
print("="*60)